# Horizon Training Dashboard
**SafeCircle Risk Detection Model** — end-to-end pipeline from data → training → evaluation

## Install Dependencies

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "python-dotenv", "pandas", "pyyaml"],
    capture_output=True, text=True
)
print(result.stdout or "Dependencies already satisfied.")
if result.returncode != 0:
    print(result.stderr)

## Environment Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import display

ROOT = Path("/home/ubuntu/horizon").resolve()

if not (ROOT / "training").exists():
    raise RuntimeError(f"Project root not found at {ROOT} — check the path.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from dotenv import load_dotenv

load_dotenv()

print(f"Project root:    {ROOT}")
print(f"data/raw exists: {(ROOT / 'data' / 'raw').exists()}")
print(f"HF_TOKEN set:    {'yes' if os.getenv('HF_TOKEN') else 'NO — set in .env'}")
print(f"BEDROCK_API_KEY: {'yes' if os.getenv('BEDROCK_API_KEY') else 'NO — set in .env'}")

def stream(cmd):
    """Run a command and stream stdout+stderr live."""
    print(f"$ {' '.join(str(c) for c in cmd)}\n")
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, cwd=ROOT) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
    if proc.returncode != 0:
        print(f"\n[exited with code {proc.returncode}]")
    return proc.returncode

## Hardware Detection

In [ ]:
import torch

has_cuda = torch.cuda.is_available()
if has_cuda:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:            {gpu_name}")
    print(f"VRAM:           {gpu_mem:.1f} GB")
    print(f"CUDA version:   {torch.version.cuda}")
    print(f"PyTorch:        {torch.__version__}")
    if gpu_mem >= 70:
        print("\n=> Recommended config: h100.yaml (rank-256 LoRA, FA2, seq 4096, batch 32)")
    elif gpu_mem >= 20:
        print("\n=> Recommended config: l4.yaml (full bfloat16, no 4-bit quantization)")
    else:
        print("\n=> Recommended config: base.yaml or quick.yaml (4-bit quantization)")
else:
    print("No GPU found — CPU training is very slow.")

## Dataset Statistics

In [ ]:
import pandas as pd

raw_dir = ROOT / "data" / "raw"
categories = ["grooming", "bullying", "sexual_content", "isolation",
              "personal_info", "platform_migration", "threats", "benign"]

print(f"Scanning: {raw_dir}")
rows = []
for cat in categories:
    f = raw_dir / f"{cat}.jsonl"
    if f.exists():
        count = sum(1 for line in open(f) if line.strip())
        size_mb = f.stat().st_size / 1e6
        rows.append({"Category": cat, "Count": count, "Size (MB)": round(size_mb, 1), "Status": "✓"})
    else:
        rows.append({"Category": cat, "Count": 0, "Size (MB)": 0.0, "Status": "missing"})

df = pd.DataFrame(rows)
total = df["Count"].sum()
print(f"Total: {total:,} conversations\n")
display(df)

## Step 0: Download Dataset from HuggingFace Hub

In [ ]:
SPLIT = "all"  # Options: "all", "processed", "raw"
stream(["python", "data/scripts/download_from_hub.py", "--split", SPLIT])

## Step 1: Preprocess Data

In [ ]:
train_path = ROOT / "data" / "processed" / "train.jsonl"
eval_path  = ROOT / "data" / "processed" / "eval.jsonl"

if train_path.exists():
    train_count = sum(1 for l in open(train_path) if l.strip())
    eval_count  = sum(1 for l in open(eval_path)  if l.strip())
    print(f"Preprocessed data found: {train_count:,} train / {eval_count:,} eval")
    print("Run the cell below only if you want to regenerate.")
else:
    print("No preprocessed data found — run the cell below.")

In [ ]:
stream(["python", "-m", "training.scripts.preprocess",
        "--input", "data/raw",
        "--output", "data/processed",
        "--split", "0.9",
        "--seed", "42"])

## Step 2: Train Model

Set `CONFIG` to one of:

| Config | Steps | Notes |
|---|---|---|
| `quick.yaml` | 500 | Fast iteration, 4-bit QLoRA |
| `base.yaml` | 10,000 | Production baseline, 4-bit QLoRA |
| `l4.yaml` | 15,000 | Full bf16, rank-128 LoRA, 24GB+ VRAM |
| `h100.yaml` | 25,000 | **H100** — rank-256 LoRA, FA2, seq 4096, batch 32 |
| `mobile.yaml` | 5,000 | MobileBERT distillation (run after full model) |

In [ ]:
CONFIG = "training/configs/h100.yaml"
RESUME = None  # e.g. "experiments/h100-20260514-102043/checkpoint-5000"

cmd = ["python", "-m", "training.scripts.train", "--config", CONFIG]
if RESUME:
    cmd += ["--resume", RESUME]

stream(cmd)

## Step 3: Evaluate Model

In [ ]:
runs = sorted((ROOT / "experiments").glob("*/final"),
              key=lambda p: p.stat().st_mtime, reverse=True) \
      if (ROOT / "experiments").exists() else []

if runs:
    print("Available checkpoints (newest first):")
    for i, r in enumerate(runs):
        print(f"  [{i}] {r}")
else:
    print("No checkpoints found. Run training first.")

In [ ]:
CHECKPOINT = str(runs[0]) if runs else ""
print(f"Evaluating: {CHECKPOINT}\n")

stream(["python", "-m", "evaluation.metrics.evaluate",
        "--checkpoint", CHECKPOINT,
        "--test-set", "data/evaluation/test.jsonl",
        "--max-samples", "200"])

## Step 3.5: Knowledge Distillation (Mobile Model)

In [ ]:
TEACHER = str(runs[0]) if runs else ""
print(f"Teacher checkpoint: {TEACHER}\n")

stream(["python", "-m", "training.scripts.distill",
        "--teacher", TEACHER,
        "--config", "training/configs/mobile.yaml"])

## Step 4: Export Mobile Model to ONNX

In [ ]:
mobile_runs = sorted((ROOT / "experiments").glob("mobile-*/final"),
                     key=lambda p: p.stat().st_mtime, reverse=True) \
              if (ROOT / "experiments").exists() else []

MOBILE_CHECKPOINT = str(mobile_runs[0]) if mobile_runs else ""
print(f"Mobile checkpoint: {MOBILE_CHECKPOINT}\n")

stream(["python", "-m", "training.scripts.export_onnx",
        "--checkpoint", MOBILE_CHECKPOINT,
        "--output", "models/mobile",
        "--quantize"])

onnx_files = list((ROOT / "models/mobile").glob("*.onnx")) \
             if (ROOT / "models/mobile").exists() else []
for f in onnx_files:
    print(f"{f.name}: {f.stat().st_size / 1e6:.1f} MB")

## Experiment History

In [ ]:
import yaml

rows = []
if (ROOT / "experiments").exists():
    for cfg_path in sorted((ROOT / "experiments").glob("*/training_config.yaml"),
                           key=lambda p: p.stat().st_mtime, reverse=True):
        try:
            cfg = yaml.safe_load(open(cfg_path))
            rows.append({
                "Run": cfg_path.parent.name,
                "Steps": cfg.get("training", {}).get("max_steps"),
                "LR": cfg.get("training", {}).get("learning_rate"),
                "Batch": cfg.get("training", {}).get("per_device_train_batch_size"),
                "LoRA rank": cfg.get("lora", {}).get("rank"),
                "Seq len": cfg.get("data", {}).get("max_seq_length"),
                "4-bit": cfg.get("quantization", {}).get("load_in_4bit"),
            })
        except Exception as e:
            print(f"Could not parse {cfg_path}: {e}")

if rows:
    display(pd.DataFrame(rows))
else:
    print("No experiments yet.")